# Model Selection

Compare five nano YOLO architectures for Net and Rope detection. The original validation split is limited, so results are exploratory.


## 1. Setup

Mount Drive, configure dataset and output paths, and set the random seed.


In [ ]:
# Reproducible Colab setup and output paths.
%pip install -q "ultralytics>=8.4,<8.5" "pandas==2.2.3" "pillow<13" pyyaml matplotlib seaborn
import os, json, random, time, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import yaml, torch, ultralytics
from PIL import Image
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Runtime > Change runtime type'
ROOT = Path('/content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed')
OUT = Path('/content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1')
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir() and (ROOT/'data.yaml').is_file(), f'Corrected dataset missing: {ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Pandas:',pd.__version__,'Ultralytics:',ultralytics.__version__)
print('Dataset:',ROOT,'Output:',OUT)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Mounted at /content/drive
GPU: Tesla T4
Pandas: 2.2.3 Ultralytics: 8.4.216
Dataset: /content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed Output: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1


## 2. Dataset checks

Verify labels, boxes, class mapping, and image counts.


In [ ]:
# Validate the existing dataset before running model comparisons.
YAML_PATH = ROOT/'data.yaml'
cfg = yaml.safe_load(YAML_PATH.read_text())
names = cfg['names']
if isinstance(names, dict): names = [names.get(i, names.get(str(i))) for i in range(2)]
assert len(names)==2 and 'net' in str(names[0]).lower() and 'rope' in str(names[1]).lower(), names
assert Path(cfg['path']).resolve()==ROOT.resolve(), f'Incorrect YAML path: {cfg["path"]}'
IMG_EXT={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
def images_for(split):
    entry=cfg.get(split)
    assert entry is not None, f'Missing {split} split'
    dirs=entry if isinstance(entry,list) else [entry]
    images=[]
    for d in dirs:
        p=Path(d); p=p if p.is_absolute() else ROOT/p
        assert p.exists(), f'Missing {split} path: {p}'
        if p.is_dir(): images += [q for q in p.rglob('*') if q.is_file() and q.suffix.lower() in IMG_EXT]
        elif p.suffix=='.txt':
            for line in p.read_text().splitlines():
                if line.strip():
                    q=Path(line.strip()); images.append(q if q.is_absolute() else p.parent/q)
        else: raise ValueError(f'Not an image directory or manifest: {p}')
    assert images and all(q.is_file() for q in images), f'Invalid {split} images'
    return sorted(set(images))
def label_for(img):
    rel=img.relative_to(ROOT/'images')
    return (ROOT/'labels'/rel).with_suffix('.txt')
def audit(split):
    images=images_for(split); counts=Counter(); missing=[]; invalid=[]; empty=0
    for im in images:
        lab=label_for(im)
        if not lab.exists(): missing.append(str(lab)); continue
        lines=[x.strip() for x in lab.read_text().splitlines() if x.strip()]
        if not lines: empty+=1
        for line in lines:
            try:
                a=line.split(); assert len(a)==5
                assert a[0] in ('0','1')
                cls=int(a[0]); x,y,w,h=map(float,a[1:])
                assert all(np.isfinite([x,y,w,h]))
                assert 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
                assert x-w/2>=-1e-5 and x+w/2<=1+1e-5
                assert y-h/2>=-1e-5 and y+h/2<=1+1e-5
                counts[cls]+=1
            except Exception: invalid.append((str(lab),line))
    return images,dict(split=split,images=len(images),nets=counts[0],ropes=counts[1],empty_labels=empty,missing_labels=len(missing),invalid_annotations=len(invalid)),missing,invalid
train_imgs,tr,tm,ti=audit('train')
val_imgs,va,vm,vi=audit('val')
display(pd.DataFrame([tr,va]))
assert not (tm or ti or vm or vi), f'Label problems: missing {len(tm)+len(vm)}, invalid {len(ti)+len(vi)}'
assert not set(train_imgs).intersection(val_imgs), 'Same image paths in train and validation'
VALIDATION_SUFFICIENT=(va['nets']>=30 and va['ropes']>=30)
if not VALIDATION_SUFFICIENT:
    print('WARNING: validation contains',va['nets'],'nets and',va['ropes'],'ropes. Exploratory training only; final selection BLOCKED.')
else: print('Minimum validation annotation count met; representativeness still needs review.')

,split,images,nets,ropes,empty_labels,missing_labels,invalid_annotations
0,train,2626,1067,2055,500,0,0
1,val,178,7,71,100,0,0


## 2.1 Duplicate checks

Check exact duplicates. Camera-level separation is not established.


In [ ]:
# Screen for exact duplicates and record the limits of provenance verification.
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
train_hash={sha(p) for p in train_imgs}
val_hash={sha(p) for p in val_imgs}
assert not (train_hash & val_hash), 'Exact train/val image duplicates found!'

# Set this only if you have independently verified separation by camera/video/site.
GROUPING_VERIFIED_BY_DATA_OWNER=False
GROUP_CSV=None # Optional CSV with columns image,group_id; covers ALL train and val images
if GROUP_CSV:
    groups=pd.read_csv(GROUP_CSV)
    assert {'image','group_id'} <= set(groups.columns)
    def ids(imgs):
        found=[]
        for im in imgs:
            matches=groups[(groups.image.astype(str)==str(im)) | (groups.image.astype(str)==im.name)]
            assert len(matches)==1, f'Missing or ambiguous group: {im}'
            found.append(str(matches.iloc[0].group_id))
        return set(found)
    assert not (ids(train_imgs)&ids(val_imgs)), 'Source group overlap!'
    GROUPING_VERIFIED_BY_DATA_OWNER=True
if not GROUPING_VERIFIED_BY_DATA_OWNER:
    print('WARNING: source grouping not verified; exploratory results only.')
else: print('Source grouping verified.')

## 3. Dataset configuration

Use the prepared dataset without changing source files.


In [ ]:
# Reuse the prepared YAML; the source dataset remains unchanged.
# Train directly against the existing fixed YAML; no manifests or dataset changes.
experiment_yaml=ROOT/'data.yaml'
assert experiment_yaml.is_file()
print('Experiment YAML:',experiment_yaml)
print(experiment_yaml.read_text())

Experiment YAML: /content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed/data.yaml
path: /content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed
train: images/train
val: images/val
nc: 2
names:
  0: Net
  1: Rope
test: images/test



## 4. Five-model screening

Screen five pretrained models with the same settings. HSV, translation, scaling, horizontal flips, and mosaic augment training images online; they do not create new dataset files.


In [ ]:
# Use a shared training configuration for all five architectures.
SPECS={'yolov8n':'yolov8n.pt','yolov10n':'yolov10n.pt','yolo11n':'yolo11n.pt','yolo12n':'yolo12n.pt','yolo26n':'yolo26n.pt'}
IMG=640; BATCH=8; WORKERS=2
# Training-only augmentation: HSV color variation, small shifts/zoom, horizontal flips, and mosaic.
# No rotation, vertical flips, or MixUp; no class-specific oversampling was applied.
COMMON=dict(data=str(experiment_yaml),imgsz=IMG,batch=BATCH,device=0,workers=WORKERS,seed=SEED,deterministic=True,
            optimizer='SGD',lr0=0.01,cos_lr=False,hsv_h=0.01,hsv_s=0.30,hsv_v=0.20,
            degrees=0,translate=0.05,scale=0.20,fliplr=0.5,flipud=0.0,mosaic=0.30,mixup=0.0,
            close_mosaic=8,plots=True,exist_ok=False,verbose=False)
RUNS=OUT/'runs_budget20_30_v1'
def train_candidate(name,stage,epochs,patience):
    run_dir=RUNS/stage/name
    best=run_dir/'weights'/'best.pt'
    if best.exists():
        print('Reusing existing checkpoint:',best)
    else:
        assert not run_dir.exists(), f'Incomplete run: {run_dir}; inspect it before retrying'
        model=YOLO(SPECS[name]); start=time.perf_counter()
        model.train(**COMMON,epochs=epochs,patience=patience,project=str(RUNS/stage),name=name)
        print(name,stage,'training seconds:',round(time.perf_counter()-start,1))
    assert best.exists(),f'No checkpoint: {best}'
    return best

def evaluate(name,stage,best):
    model=YOLO(str(best))
    m=model.val(data=str(experiment_yaml),split='val',imgsz=IMG,batch=BATCH,device=0,
                plots=True,project=str(RUNS/'evaluation'/stage),name=name,exist_ok=True,verbose=False)
    # Align Ultralytics per-class metrics to the corresponding class IDs.
    per={int(c): (float(m.box.r[i]),float(m.box.p[i]),float(m.box.maps[int(c)])) for i,c in enumerate(m.box.ap_class_index)}
    row={'model':name,'stage':stage,'checkpoint':str(best),'map50_95':float(m.box.map),'map50':float(m.box.map50)}
    for cls,label in [(0,'nets'),(1,'ropes')]:
        assert cls in per,f'No evaluation metrics for {label}'
        row[f'{label}_recall'],row[f'{label}_precision'],row[f'{label}_map50_95']=per[cls]
    row['min_recall']=min(row['nets_recall'],row['ropes_recall'])
    row['macro_recall']=(row['nets_recall']+row['ropes_recall'])/2
    row['macro_map']=(row['nets_map50_95']+row['ropes_map50_95'])/2
    return row

screen_rows=[]
for name in SPECS:
    try:
        best=train_candidate(name,'screening',20,6)
        screen_rows.append(evaluate(name,'screening',best))
    except Exception as e:
        print('FAILED:',name,repr(e))
    pd.DataFrame(screen_rows).to_csv(OUT/'screening_results.csv',index=False)
screen=pd.DataFrame(screen_rows)
assert len(screen)==len(SPECS),'Not all five models completed; fix failures before choosing finalists.'
display(screen.sort_values(['min_recall','macro_map'],ascending=False))

Reusing existing checkpoint: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1/runs_budget20_30_v1/screening/yolov8n/weights/best.pt
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.4 ms, read: 88.5±25.5 MB/s, size: 251.4 KB)
val: Scanning /content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed/labels/val.cache... 178 images, 100 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 178/178 39.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 6.4it/s 3.6s
                   all        178         78     0.0841     0.0704     0.0607     0.0288
Speed: 1.0ms preprocess, 5.2ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1/runs_budget20_30_v1/evaluation/screening/yolov

,model,stage,checkpoint,map50_95,map50,nets_recall,nets_precision,nets_map50_95,ropes_recall,ropes_precision,ropes_map50_95,min_recall,macro_recall,macro_map
3,yolo12n,screening,/content/drive/MyDrive/MarineGuard_Model_Selec...,0.260408,0.560811,1.000000,0.534887,0.367117,0.361841,0.837047,0.153699,0.361841,0.680921,0.260408
4,yolo26n,screening,/content/drive/MyDrive/MarineGuard_Model_Selec...,0.237839,0.477132,0.142857,0.290497,0.157196,0.591549,0.734275,0.318482,0.142857,0.367203,0.237839
2,yolo11n,screening,/content/drive/MyDrive/MarineGuard_Model_Selec...,0.157571,0.325726,0.000000,0.000000,0.003934,0.619718,0.591320,0.311209,0.000000,0.309859,0.157571
1,yolov10n,screening,/content/drive/MyDrive/MarineGuard_Model_Selec...,0.033615,0.083294,0.000000,0.000000,0.000353,0.366197,0.192813,0.066877,0.000000,0.183099,0.033615
0,yolov8n,screening,/content/drive/MyDrive/MarineGuard_Model_Selec...,0.028819,0.060732,0.000000,0.000000,0.000000,0.140845,0.168196,0.057638,0.000000,0.070423,0.028819


## 5. Finalist training

Train two finalists independently for up to 30 epochs.


In [ ]:
# Start both finalists from pretrained weights, not screening checkpoints.
TOP_K=2
finalists=screen.sort_values(['min_recall','macro_map'],ascending=False).head(TOP_K)['model'].tolist()
print('Finalists:',finalists)
full_rows=[]
for name in finalists:
    try:
        # Independent pretrained initialization; 30-epoch maximum.
        best=train_candidate(name,'final',30,8)
        full_rows.append(evaluate(name,'final',best))
    except Exception as e:
        print('FAILED:',name,repr(e))
    pd.DataFrame(full_rows).to_csv(OUT/'full_results.csv',index=False)
full=pd.DataFrame(full_rows)
assert len(full)==len(finalists),'Finalist run failed; do not select a winner yet.'
display(full.sort_values(['min_recall','macro_map'],ascending=False))

Finalists: ['yolo12n', 'yolo26n']
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed/data.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.3, hsv_v=0.2, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=0.3, multi_sc

## 6. Confidence thresholds

Measure precision, recall and F1 across validation confidence thresholds.


In [ ]:

# Validation threshold sweep | YOLO12n and YOLO26n | CPU | IoU >= 0.50

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO
from collections import defaultdict
import time

drive.mount("/content/drive")

# Checkpoint paths and evaluation settings.

ROOT = Path(
    "/content/drive/MyDrive/MarineGuard_YOLO/"
    "MarineGuard_YOLO_Fixed"
)

OUT = Path(
    "/content/drive/MyDrive/MarineGuard_Model_Selection/"
    "budget20_30_v1"
)

RUNS = OUT / "runs_budget20_30_v1"
VAL_DIR = ROOT / "images" / "val"

IMG = 640
DEVICE = "cpu"
THRESHOLDS = [0.05, 0.10, 0.15, 0.20,
              0.25, 0.30, 0.40, 0.50]
IOU_THRESHOLD = 0.50

IMG_EXT = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

assert VAL_DIR.is_dir(), f"Missing: {VAL_DIR}"

val_imgs = sorted(
    p for p in VAL_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXT
)

assert val_imgs, "No validation images found."

def label_for(img):
    relative = Path(img).relative_to(ROOT / "images")
    return (ROOT / "labels" / relative).with_suffix(".txt")

CHECKPOINTS = {
    "yolo12n": RUNS / "final/yolo12n/weights/best.pt",
    "yolo26n": RUNS / "final/yolo26n/weights/best.pt"
}

for name, path in CHECKPOINTS.items():
    assert path.is_file(), f"Missing checkpoint: {path}"

OUT.mkdir(parents=True, exist_ok=True)

print("Validation images:", len(val_imgs))
print("Device:", DEVICE)
print("Models:", list(CHECKPOINTS))
print("Output:", OUT)

# Load ground-truth annotations as normalized [class, x1, y1, x2, y2] boxes.

def gt_boxes(img_path):
    label_path = label_for(img_path)

    if not label_path.exists():
        # Treat absent annotations as no labeled objects for this audit.
        return np.empty((0, 5), dtype=float)

    with Image.open(img_path) as im:
        width, height = im.size

    boxes = []

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue

        values = line.split()

        if len(values) != 5:
            raise ValueError(f"Invalid label: {label_path}")

        cls = int(values[0])
        x, y, w, h = map(float, values[1:])

        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2

        boxes.append([cls, x1, y1, x2, y2])

    return np.array(boxes, dtype=float).reshape(-1, 5)


TRUTH = [gt_boxes(p) for p in val_imgs]

print("Ground-truth labels loaded.")

# Bounding-box intersection over union.

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = max(0, box1[2] - box1[0]) * max(
        0, box1[3] - box1[1]
    )
    area2 = max(0, box2[2] - box2[0]) * max(
        0, box2[3] - box2[1]
    )

    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

# Match detections to ground truth, one-to-one, in descending confidence order.

def evaluate_image(gt, pred, cls_id, threshold):
    gt_cls = gt[gt[:, 0] == cls_id]
    pred_cls = pred[
        (pred[:, 0] == cls_id) &
        (pred[:, 1] >= threshold)
    ]

    if len(pred_cls):
        pred_cls = pred_cls[
            np.argsort(-pred_cls[:, 1])
        ]

    matched_gt = set()
    tp = 0
    fp = 0

    for prediction in pred_cls:
        pred_box = prediction[2:6]

        best_iou = 0.0
        best_idx = -1

        for j, ground_truth in enumerate(gt_cls):
            if j in matched_gt:
                continue

            iou = calculate_iou(
                pred_box,
                ground_truth[1:5]
            )

            if iou > best_iou:
                best_iou = iou
                best_idx = j

        if best_iou >= IOU_THRESHOLD and best_idx >= 0:
            matched_gt.add(best_idx)
            tp += 1
        else:
            fp += 1

    fn = len(gt_cls) - tp

    return tp, fp, fn

# Cache model predictions once; calculate TP, FP, and FN at each threshold.

all_rows = []

for model_name, checkpoint in CHECKPOINTS.items():
    print(f"\nRunning {model_name} on CPU...")
    start = time.time()

    model = YOLO(str(checkpoint))

    predictions = []

    for idx, img_path in enumerate(val_imgs):
        result = model.predict(
            source=str(img_path),
            imgsz=IMG,
            conf=0.001,
            iou=0.70,
            max_det=300,
            device=DEVICE,
            verbose=False
        )[0]

        if result.boxes is None or len(result.boxes) == 0:
            pred = np.empty((0, 6), dtype=float)
        else:
            classes = result.boxes.cls.cpu().numpy()
            confidences = result.boxes.conf.cpu().numpy()
            boxes = result.boxes.xyxyn.cpu().numpy()

            pred = np.column_stack([
                classes,
                confidences,
                boxes
            ])

        predictions.append(pred)

        if (idx + 1) % 25 == 0:
            print(f"  Processed {idx + 1}/{len(val_imgs)} images")

    elapsed = time.time() - start
    print(f"Inference completed in {elapsed:.1f} seconds")

    # Evaluate every confidence threshold
    for threshold in THRESHOLDS:
        for cls_id, class_name in [(0, "Net"), (1, "Rope")]:
            TP = FP = FN = 0
            negative_fp = 0

            for gt, pred in zip(TRUTH, predictions):
                tp, fp, fn = evaluate_image(
                    gt, pred, cls_id, threshold
                )

                TP += tp
                FP += fp
                FN += fn

                # Track false positives on fully negative images.
                if len(gt) == 0:
                    negative_fp += fp

            precision = TP / (TP + FP) if TP + FP else 0.0
            recall = TP / (TP + FN) if TP + FN else 0.0

            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )

            all_rows.append({
                "model": model_name,
                "threshold": threshold,
                "class": class_name,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "negative_fp": negative_fp,
                "inference_seconds": elapsed
            })

    # Preserve completed results after each model.
    pd.DataFrame(all_rows).to_csv(
        OUT / "threshold_sweep.csv",
        index=False
    )

    print(f"Saved threshold results for {model_name}")

# Summarize per-class thresholds and export the highest-F1 operating points.

threshold_results = pd.DataFrame(all_rows)

print("\n=== CONFIDENCE THRESHOLD RESULTS ===")
display(threshold_results)

# Choose the highest measured class-specific F1 on validation.
best_thresholds = (
    threshold_results
    .sort_values(
        ["f1", "recall"],
        ascending=[False, False]
    )
    .drop_duplicates(["model", "class"])
    .sort_values(["class", "model"])
)

print("\n=== BEST F1 THRESHOLD PER MODEL AND CLASS ===")
display(best_thresholds[
    [
        "model", "class", "threshold",
        "precision", "recall", "f1",
        "TP", "FP", "FN", "negative_fp"
    ]
])

best_thresholds.to_csv(
    OUT / "best_thresholds.csv",
    index=False
)

print("\nAnalysis completed.")
print("Saved:", OUT / "threshold_sweep.csv")
print("Saved:", OUT / "best_thresholds.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Validation images: 178
Device: cpu
Models: ['yolo12n', 'yolo26n']
Output: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1
Ground-truth labels loaded.

Running yolo12n on CPU...
  Processed 25/178 images
  Processed 50/178 images
  Processed 75/178 images
  Processed 100/178 images
  Processed 125/178 images
  Processed 150/178 images
  Processed 175/178 images
Inference completed in 23.7 seconds
Saved threshold results for yolo12n

Running yolo26n on CPU...
  Processed 25/178 images
  Processed 50/178 images
  Processed 75/178 images
  Processed 100/178 images
  Processed 125/178 images
  Processed 150/178 images
  Processed 175/178 images
Inference completed in 14.0 seconds
Saved threshold results for yolo26n

=== CONFIDENCE THRESHOLD RESULTS ===


,model,threshold,class,TP,FP,FN,precision,recall,f1,negative_fp,inference_seconds
0,yolo12n,0.05,Net,7,77,0,0.083333,1.000000,0.153846,8,23.670773
1,yolo12n,0.05,Rope,40,331,31,0.107817,0.563380,0.180995,52,23.670773
2,yolo12n,0.10,Net,7,42,0,0.142857,1.000000,0.250000,3,23.670773
3,yolo12n,0.10,Rope,30,144,41,0.172414,0.422535,0.244898,32,23.670773
4,yolo12n,0.15,Net,7,27,0,0.205882,1.000000,0.341463,3,23.670773
5,yolo12n,0.15,Rope,24,74,47,0.244898,0.338028,0.284024,20,23.670773
6,yolo12n,0.20,Net,7,12,0,0.368421,1.000000,0.538462,3,23.670773
7,yolo12n,0.20,Rope,21,39,50,0.350000,0.295775,0.320611,9,23.670773
8,yolo12n,0.25,Net,6,10,1,0.375000,0.857143,0.521739,2,23.670773
9,yolo12n,0.25,Rope,16,16,55,0.500000,0.225352,0.310680,1,23.670773



=== BEST F1 THRESHOLD PER MODEL AND CLASS ===


,model,class,threshold,precision,recall,f1,TP,FP,FN,negative_fp
10,yolo12n,Net,0.30,0.666667,0.571429,0.615385,4,2,3,1
24,yolo26n,Net,0.25,0.428571,0.857143,0.571429,6,8,1,0
7,yolo12n,Rope,0.20,0.350000,0.295775,0.320611,21,39,50,9
21,yolo26n,Rope,0.15,0.646154,0.591549,0.617647,42,23,29,3



Analysis completed.
Saved: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1/threshold_sweep.csv
Saved: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1/best_thresholds.csv


## 7. CPU inference speed

Benchmark the saved finalists on matching images.


In [ ]:

# CPU inference benchmark | YOLO12n and YOLO26n | 640px

from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import time

ROOT = Path(
    "/content/drive/MyDrive/MarineGuard_YOLO/"
    "MarineGuard_YOLO_Fixed"
)

OUT = Path(
    "/content/drive/MyDrive/MarineGuard_Model_Selection/"
    "budget20_30_v1"
)

RUNS = OUT / "runs_budget20_30_v1"

CHECKPOINTS = {
    "yolo12n": RUNS / "final/yolo12n/weights/best.pt",
    "yolo26n": RUNS / "final/yolo26n/weights/best.pt"
}

IMG = 640
DEVICE = "cpu"

IMG_EXT = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

val_imgs = sorted(
    p for p in (ROOT / "images/val").rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXT
)

assert val_imgs, "No validation images found."

sample = val_imgs[:min(30, len(val_imgs))]

bench = []

for name, checkpoint in CHECKPOINTS.items():

    assert checkpoint.exists(), f"Missing: {checkpoint}"

    print(f"\nBenchmarking {name} on CPU...")

    model = YOLO(str(checkpoint))

    # Exclude warm-up predictions from measured latency.
    for img in sample[:5]:
        model.predict(
            source=str(img),
            imgsz=IMG,
            device=DEVICE,
            verbose=False
        )

    # Time inference over the common validation-image sample.
    t0 = time.perf_counter()

    for img in sample:
        model.predict(
            source=str(img),
            imgsz=IMG,
            device=DEVICE,
            verbose=False
        )

    elapsed = time.perf_counter() - t0

    bench.append({
        "model": name,
        "device": DEVICE,
        "images_tested": len(sample),
        "latency_ms_per_image": 1000 * elapsed / len(sample),
        "fps": len(sample) / elapsed,
        "checkpoint_mb": checkpoint.stat().st_size / 1e6
    })

bench = pd.DataFrame(bench)

bench.to_csv(
    OUT / "benchmark_cpu.csv",
    index=False
)

print("\n=== CPU BENCHMARK RESULTS ===")
display(bench)

print("\nSaved:", OUT / "benchmark_cpu.csv")



Benchmarking yolo12n on CPU...

Benchmarking yolo26n on CPU...

=== CPU BENCHMARK RESULTS ===


,model,device,images_tested,latency_ms_per_image,fps,checkpoint_mb
0,yolo12n,cpu,30,86.746162,11.527888,5.529754
1,yolo26n,cpu,30,67.847352,14.738969,10.641633



Saved: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1/benchmark_cpu.csv


## 8. Checkpoint evaluation

Recalculate per-class validation metrics without retraining.


In [ ]:

# Recompute saved-finalist validation metrics on CPU; no training.

from pathlib import Path
import pandas as pd
from ultralytics import YOLO

OUT = Path(
    "/content/drive/MyDrive/MarineGuard_Model_Selection/"
    "budget20_30_v1"
)

RUNS = OUT / "runs_budget20_30_v1"

DATA = (
    "/content/drive/MyDrive/MarineGuard_YOLO/"
    "MarineGuard_YOLO_Fixed/data.yaml"
)

CHECKPOINTS = {
    "yolo12n": RUNS / "final/yolo12n/weights/best.pt",
    "yolo26n": RUNS / "final/yolo26n/weights/best.pt"
}

rows = []

for name, checkpoint in CHECKPOINTS.items():
    assert checkpoint.is_file(), f"Missing checkpoint: {checkpoint}"

    print(f"\nEvaluating {name} on CPU...")

    model = YOLO(str(checkpoint))

    metrics = model.val(
        data=DATA,
        split="val",
        imgsz=640,
        batch=4,
        device="cpu",
        workers=0,
        plots=False,
        verbose=False
    )

    per_class = {}

    for i, cls in enumerate(metrics.box.ap_class_index):
        cls = int(cls)
        per_class[cls] = {
            "precision": float(metrics.box.p[i]),
            "recall": float(metrics.box.r[i]),
            "map50_95": float(metrics.box.maps[cls])
        }

    row = {
        "model": name,
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map)
    }

    for cls, label in [(0, "nets"), (1, "ropes")]:
        values = per_class.get(cls, {})
        row[f"{label}_precision"] = values.get("precision", float("nan"))
        row[f"{label}_recall"] = values.get("recall", float("nan"))
        row[f"{label}_map50_95"] = values.get("map50_95", float("nan"))

    rows.append(row)

    # Export partial results after each completed evaluation.
    pd.DataFrame(rows).to_csv(
        OUT / "finalists_cpu_comparison.csv",
        index=False
    )

comparison = pd.DataFrame(rows)

print("\n=== FINALIST COMPARISON ===")
display(comparison)

print("\nSaved:", OUT / "finalists_cpu_comparison.csv")



Evaluating yolo12n on CPU...
Ultralytics 8.4.216 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (AMD EPYC 7B12)
YOLOv12n summary (fused): 160 layers, 2,571,870 parameters, 0 gradients, 7.4 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.3 ms, read: 151.0±62.2 MB/s, size: 150.5 KB)
val: Scanning /content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed/labels/val.cache... 178 images, 100 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 178/178 29.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 2.7it/s 16.9s
                   all        178         78      0.352      0.527      0.444       0.25
Speed: 0.7ms preprocess, 83.4ms inference, 0.0ms loss, 0.4ms postprocess per image

Evaluating yolo26n on CPU...
Ultralytics 8.4.216 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (AMD EPYC 7B12)
YOLO26n summary (fused): 121 layers, 2,389,978 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 265.0±105.5 MB

,model,map50,map50_95,nets_precision,nets_recall,nets_map50_95,ropes_precision,ropes_recall,ropes_map50_95
0,yolo12n,0.444107,0.249543,0.394430,0.857143,0.436551,0.309158,0.197183,0.062535
1,yolo26n,0.533364,0.327078,0.450133,0.857143,0.299444,0.944321,0.477782,0.354711



Saved: /content/drive/MyDrive/MarineGuard_Model_Selection/budget20_30_v1/finalists_cpu_comparison.csv


## 9. External evaluation

Run optional external testing without using its scores for initial selection.


In [ ]:

# Optional external evaluation of the selected checkpoint; no training.

from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
import ultralytics
import pandas as pd
import torch
import json
import yaml

drive.mount("/content/drive")

# Dataset path and evaluation settings.

OUT = Path(
    "/content/drive/MyDrive/MarineGuard_Model_Selection/"
    "budget20_30_v1"
)

IMG = 640
DEVICE = "cpu"

# Keep evaluation disabled until a labeled test YAML is supplied.
EXTERNAL_TEST_YAML = None

SELECTION_FILE = OUT / "PROVISIONAL_SELECTION.json"

# Skip external evaluation when the test configuration is absent.

if EXTERNAL_TEST_YAML is None:
    print("External test skipped.")
    print("No independent test dataset has been configured.")
    print("Your saved checkpoints and results are unchanged.")

else:
        # Verify class ordering and external split configuration.

    external_yaml = Path(EXTERNAL_TEST_YAML)

    assert external_yaml.is_file(), (
        f"External dataset YAML not found: {external_yaml}"
    )

    with external_yaml.open("r", encoding="utf-8") as f:
        ext_cfg = yaml.safe_load(f)

    assert isinstance(ext_cfg, dict), (
        "Invalid external dataset configuration."
    )

    ext_names = ext_cfg.get("names")

    assert ext_names is not None, (
        "External dataset YAML must define class names."
    )

    if isinstance(ext_names, dict):
        ext_names = [
            ext_names.get(i, ext_names.get(str(i)))
            for i in range(2)
        ]

    assert len(ext_names) == 2, (
        "External dataset must contain exactly two classes."
    )

    assert "net" in str(ext_names[0]).lower(), (
        "Class 0 must represent nets."
    )

    assert "rope" in str(ext_names[1]).lower(), (
        "Class 1 must represent ropes."
    )

    assert ext_cfg.get("test"), (
        "External dataset must define a test split."
    )

        # Read the checkpoint and operating threshold from the selection record.

    assert SELECTION_FILE.is_file(), (
        "Provisional selection not found. "
        "Complete the model-selection step first."
    )

    with SELECTION_FILE.open("r", encoding="utf-8") as f:
        selection = json.load(f)

    checkpoint = Path(selection["checkpoint"])
    model_name = selection["model"]
    threshold = float(selection["threshold"])

    assert checkpoint.is_file(), (
        f"Selected checkpoint not found: {checkpoint}"
    )

    print("=== EXTERNAL TEST CONFIGURATION ===")
    print("Selected model:", model_name)
    print("Checkpoint:", checkpoint)
    print("Confidence threshold:", threshold)
    print("Device:", DEVICE)
    print("Dataset:", external_yaml)

        # Compute standard detection metrics at low confidence.

    model = YOLO(str(checkpoint))

    metrics = model.val(
        data=str(external_yaml),
        split="test",
        imgsz=IMG,
        batch=4,
        device=DEVICE,
        workers=0,
        conf=0.001,
        plots=True,
        project=str(OUT / "external_test"),
        name="provisional_model",
        exist_ok=False,
        verbose=False
    )

        # Collect overall and class-specific performance.

    results = {
        "model": model_name,
        "checkpoint": str(checkpoint),
        "selected_threshold": threshold,
        "evaluation_confidence": 0.001,
        "device": DEVICE,
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "ultralytics_version": ultralytics.__version__
    }

    for i, cls in enumerate(metrics.box.ap_class_index):
        cls = int(cls)
        label = {0: "nets", 1: "ropes"}.get(
            cls, f"class_{cls}"
        )

        results[f"{label}_precision"] = float(
            metrics.box.p[i]
        )

        results[f"{label}_recall"] = float(
            metrics.box.r[i]
        )

        results[f"{label}_map50_95"] = float(
            metrics.box.maps[cls]
        )

        # Save the external evaluation record without changing model weights.

    output_dir = OUT / "external_test"
    output_dir.mkdir(parents=True, exist_ok=True)

    results_file = output_dir / "external_test_results.json"

    results_file.write_text(
        json.dumps(results, indent=2),
        encoding="utf-8"
    )

    pd.DataFrame([results]).to_csv(
        output_dir / "external_test_results.csv",
        index=False
    )

    print("\n=== EXTERNAL TEST RESULTS ===")
    display(pd.DataFrame([results]))

    print("\nResults saved:")
    print(results_file)
    print(output_dir / "external_test_results.csv")

    print(
        "\nNote: The mAP metrics use confidence 0.001 "
        "for evaluation. The selected deployment threshold "
        "is recorded but not evaluated as fixed-threshold "
        "TP/FP/FN in this cell."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
External test skipped.
No independent test dataset has been configured.
Your saved checkpoints and results are unchanged.


## 10. Outcome

The original YOLO26n checkpoint was retained for Version 1. Its finalist run recorded 21 epochs; the validation set contained seven Nets. Later notebooks investigate dataset limitations.
